# Day 44 — Solutions: Model Deployment with FastAPI
Train + save model (joblib), serve predictions via FastAPI, test with curl.

In [ ]:
from pathlib import Path

import joblib
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

artifact_dir = Path('artifacts/day44')
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / 'model.joblib'

X, y = load_iris(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
clf.score(Xte, yte), joblib.dump(clf, model_path)

## Minimal FastAPI app (save as app.py)

In [ ]:
app_py = '''
from typing import Annotated

from pathlib import Path

import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel, Field

artifact_dir = Path(__file__).resolve().parent
app = FastAPI()
model = joblib.load(artifact_dir / 'model.joblib')
CLASS_NAMES = ['setosa', 'versicolor', 'virginica']

class IrisFeatures(BaseModel):
    features: Annotated[list[float], Field(min_length=4, max_length=4)]

@app.post('/predict')
def predict(data: IrisFeatures):
    X = np.array([data.features], dtype=float)
    pred = int(model.predict(X).tolist()[0])
    return {'prediction': pred, 'class_name': CLASS_NAMES[pred]}
'''

app_path = artifact_dir / 'app.py'
app_path.write_text(app_py, encoding='utf-8')
f'Wrote {app_path}'


## Run server (in terminal)
uvicorn app:app --reload

## Test with curl
curl -s -X POST http://127.0.0.1:8000/predict -H 'Content-Type: application/json' -d '{"features": [5.1, 3.5, 1.4, 0.2]}'